# Deteksi Anomali Titik Panas Karhutla Jawa Barat
Isolation Forest dengan pembanding sederhana. Data: NASA FIRMS VIIRS archive (NOAA-20, S-NPP, NOAA-21), batas wilayah GADM 4.1 level 1.

**Sebelum menjalankan:** upload ke panel Files: `fire_archive_J1V-C2_836775.csv`, `fire_archive_SV-C2_836777.csv`, `fire_archive_J2V-C2_836776.csv`, dan `jabar_batas.zip`.


## 1. Instal library dan ekstrak batas wilayah

In [1]:
!pip install pyshp shapely -q

import zipfile, glob
zipfile.ZipFile('jabar_batas.zip').extractall('batas')
print(glob.glob('batas/**/*.shp', recursive=True))


['batas/gadm41_IDN_1.shp']


## 2. Gabung tiga satelit, potong batas Jawa Barat, dan filter

In [2]:
import pandas as pd, numpy as np
import shapefile
from shapely.geometry import shape, Point
from shapely.prepared import prep

files = {'NOAA-20': 'fire_archive_J1V-C2_836775.csv',
         'S-NPP':   'fire_archive_SV-C2_836777.csv',
         'NOAA-21': 'fire_archive_J2V-C2_836776.csv'}
raw = pd.concat([pd.read_csv(f).assign(src=s) for s, f in files.items()],
                ignore_index=True)
print('Data mentah:', len(raw))

# Potong batas Jawa Barat (dilakukan SEBELUM fitur dihitung)
r = shapefile.Reader(glob.glob('batas/**/*.shp', recursive=True)[0])
i = [k for k, x in enumerate(r.records()) if x['NAME_1'] == 'Jawa Barat'][0]
P = prep(shape(r.shape(i).__geo_interface__))
inside = [P.contains(Point(lo, la)) for lo, la in zip(raw.longitude, raw.latitude)]
df = raw[inside].copy()
print('Setelah potong Jawa Barat:', len(df))

# Filter sumber dan confidence
df = df[(df['type'] == 0) & (df.confidence.isin(['n', 'h']))].copy()
df['date'] = pd.to_datetime(df.acq_date)
print('Setelah filter:', len(df))


Data mentah: 32986
Setelah potong Jawa Barat: 25093
Setelah filter: 17469


## 3. Hapus duplikat antar satelit dan bentuk fitur spatio-temporal

In [3]:
# Duplikat: lokasi dibulatkan 0,01 derajat dan tanggal sama, simpan FRP tertinggi
df['lat2'] = df.latitude.round(2); df['lon2'] = df.longitude.round(2)
df = (df.sort_values('frp', ascending=False)
        .drop_duplicates(['lat2', 'lon2', 'acq_date']))
print('Setelah hapus duplikat:', len(df))

# Jumlah titik per sel 0,1 derajat dalam 7 hari terakhir
df['cell'] = (np.floor(df.latitude*10).astype(int).astype(str) + '_' +
              np.floor(df.longitude*10).astype(int).astype(str))
full = pd.date_range(df.date.min(), df.date.max())
daily = (df.groupby(['cell', 'date']).size()
           .unstack(fill_value=0).reindex(columns=full, fill_value=0))
roll = daily.T.rolling(7, min_periods=1).sum().T
st = roll.stack().rename('count_7d').reset_index()
st.columns = ['cell', 'date', 'count_7d']
df = df.merge(st, on=['cell', 'date'], how='left')
df['daynight_d'] = (df.daynight == 'D').astype(int)


Setelah hapus duplikat: 13687


## 4. Isolation Forest, pembanding, dan simpan hasil

In [4]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

FEATURES = ['latitude', 'longitude', 'frp', 'brightness',
            'bright_t31', 'count_7d', 'daynight_d']
X = df[FEATURES].values

# Metode utama
model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42).fit(X)
df['label'] = model.predict(X)
df['score'] = -model.score_samples(X)
if_lab = df.label.values == -1

# Pembanding 1: persentil 95 pada count_7d
p95 = (df.count_7d > np.percentile(df.count_7d, 95)).values
# Pembanding 2: z-score per sel > 2
g = df.groupby('cell').count_7d
z = (df.count_7d - g.transform('mean')) / g.transform('std').replace(0, np.nan)
zs = (z > 2).fillna(False).values
# Pembanding 3: LOF
lof = LocalOutlierFactor(n_neighbors=35, contamination=0.05).fit_predict(X) == -1

bulan = df.date.dt.month.isin([8, 9, 10]).values
rows = []
for name, m in {'IsolationForest': if_lab, 'Persentil95_count7d': p95,
                'Zscore_per_sel_>2': zs, 'LOF': lof}.items():
    rows.append({'metode': name, 'jumlah_anomali': int(m.sum()),
                 '%_anomali_Agu_Okt': round(bulan[m].mean()*100, 1),
                 'overlap_dengan_IF_%': round((m & if_lab).sum() /
                                              max((m | if_lab).sum(), 1)*100, 1)})
tabel = pd.DataFrame(rows)
tabel.to_csv('perbandingan_metode_karhutla.csv', index=False)

out = df.copy()
out['date'] = out.date.dt.strftime('%Y-%m-%d')
out['label'] = out.label.map({-1: 'anomali', 1: 'normal'})
cols = ['date','latitude','longitude','brightness','bright_t31','frp',
        'daynight','confidence','src','count_7d','label','score']
out[cols].sort_values('score', ascending=False).to_csv('anomali_jabar_final_c005.csv', index=False)

df[['date','latitude','longitude','brightness','bright_t31','frp','daynight',
    'confidence','src','count_7d']].assign(date=df.date.dt.strftime('%Y-%m-%d')) \
    .to_csv('2_data_olahan_titik_jabar.csv', index=False)

print(tabel.to_string(index=False))


             metode  jumlah_anomali  %_anomali_Agu_Okt  overlap_dengan_IF_%
    IsolationForest             685               75.5                100.0
Persentil95_count7d             583               93.0                 27.3
  Zscore_per_sel_>2             649               83.1                 12.1
                LOF             685               70.8                 21.6
